# Content Portfolio Playbook PDF Generator
Produces the one-page PDF deliverable using ReportLab.

In [1]:
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import letter
from reportlab.lib.utils import ImageReader

PAGE_W, PAGE_H = letter
MARGIN = 36
CONTENT_W = PAGE_W - (MARGIN * 2)

# Brand palette
PRIMARY    = (0.106, 0.176, 0.306)   # headlines
BG         = (0.965, 0.933, 0.890)   # background
ACCENT     = (0.776, 0.365, 0.227)   # CTAs and highlights
TEXT       = (0.165, 0.145, 0.125)   # body copy
SECONDARY  = (0.561, 0.627, 0.541)   # dividers and emphasis
WHITE      = (1.0,   1.0,   1.0)

print(f"Page: {PAGE_W} x {PAGE_H} pts")
print(f"Usable width: {CONTENT_W} pts")

Page: 612.0 x 792.0 pts
Usable width: 540.0 pts


In [91]:
c = canvas.Canvas("content_portfolio_playbook.pdf", pagesize=letter)

# ── Background ───────────────────────────────────────────────────────
c.setFillColorRGB(*BG)
c.rect(0, 0, PAGE_W, PAGE_H, fill=1, stroke=0)

# ── Title block ──────────────────────────────────────────────────────
c.setFillColorRGB(*PRIMARY)
c.setFont("Helvetica-Bold", 22)
c.drawCentredString(PAGE_W / 2, 748, "Content Portfolio Playbook")

c.setFillColorRGB(*SECONDARY)
c.setFont("Helvetica", 11)
c.drawCentredString(PAGE_W / 2, 728, "Cancellation Risk × Engagement Signal → Portfolio Action")

c.setStrokeColorRGB(*SECONDARY)
c.setLineWidth(0.5)
c.line(MARGIN, 700, PAGE_W - MARGIN, 700)

# ── Matrix image ─────────────────────────────────────────────────────
MATRIX_W = CONTENT_W
MATRIX_H = 310
MATRIX_Y = 690 - MATRIX_H   # sits flush below the divider

c.drawImage("visualizations/12_portfolio_matrix.png",
            MARGIN, MATRIX_Y,
            width=MATRIX_W, height=MATRIX_H,
            preserveAspectRatio=True, anchor='nw')

# ── How to read panel (right of matrix) ──────────────────────────────
PANEL_X = MARGIN + 405
PANEL_W = CONTENT_W - MATRIX_W
PANEL_Y = 650 - 20

panel_lines = [
    ("How to Use This Playbook", True, PRIMARY, 9),
    ("", False, TEXT, 7),
    ("Each cell maps a show's risk and", False, TEXT, 7),
    ("engagement to a concrete portfolio", False, TEXT, 7),
    ("action.", False, TEXT, 7),
    ("", False, TEXT, 7),
    ("Risk score", True, ACCENT, 7),
    ("XGBoost cancellation probability (0–1).", False, TEXT, 7),
    ("", False, TEXT, 7),
    ("Engagement score", True, ACCENT, 7),
    ("Proxy: vote count + popularity + seasons", False, TEXT, 7),
    ("(median split).", False, TEXT, 7),
    ("", False, TEXT, 7),
    ("", False, TEXT, 7),
    ("Use the action label to drive your Monday", False, TEXT, 7),
    ("renewal decision.", False, TEXT, 7),
]

y = PANEL_Y
for text, bold, color, size in panel_lines:
    c.setFillColorRGB(*color)
    c.setFont("Helvetica-Bold" if bold else "Helvetica", size)
    c.drawString(PANEL_X, y, text)
    y -= 11


# ── Divider below matrix ──────────────────────────────────────────────
c.setStrokeColorRGB(*SECONDARY)
c.line(MARGIN, MATRIX_Y - 30, PAGE_W - MARGIN, MATRIX_Y - 30)

# ── Bottom section: three columns ────────────────────────────────────
COL_TOP = MATRIX_Y - 70
SHAP_H  = 190

COL1_X = MARGIN
COL2_X = MARGIN + int(CONTENT_W *0.40)
COL3_X = MARGIN + int(CONTENT_W * 0.75)
COL_W  = int(CONTENT_W * 0.27)

# ── SHAP section label ───────────────────────────────────────────────
SHAP_TOP = COL_TOP
c.setFillColorRGB(*PRIMARY)
c.setFont("Helvetica-Bold", 10)
c.drawString(MARGIN, SHAP_TOP, "Top Cancellation Risk Drivers")

c.setFillColorRGB(*SECONDARY)
c.setFont("Helvetica", 8)

# ── SHAP bar chart ───────────────────────────────────────────────────
SHAP_H = 190
SHAP_Y = SHAP_TOP - 15 - SHAP_H
c.drawImage("visualizations/09_shap_summary_bar.png",
            MARGIN, SHAP_Y, width=CONTENT_W * 0.55, height=SHAP_H,
            preserveAspectRatio=True, anchor='nw')

# ── Column 2: Key insights ───────────────────────────────────────────
c.setFillColorRGB(*PRIMARY)
c.setFont("Helvetica-Bold", 10)
c.drawString(COL2_X, COL_TOP, "Key Insights")

insights = [
    ("Seasons signal survival", True, TEXT, 8),
    ("More seasons = lower cancellation risk.", False, SECONDARY, 7),
    ("", False, TEXT, 7),
    ("Popularity beats ratings", True, TEXT, 8),
    ("Engagement volume outweighs raw score.", False, SECONDARY, 7),
    ("", False, TEXT, 7),
    ("Genre is a secondary factor", True, TEXT, 8),
    ("Sci-Fi & Comedy carry highest genre risk.", False, SECONDARY, 7),
    ("", False, TEXT, 7),
    ("Recency raises risk", True, TEXT, 8),
    ("Newer shows face higher early cancellation.", False, SECONDARY, 7),
]

y = COL_TOP - 28
for text, bold, color, size in insights:
    c.setFillColorRGB(*color)
    c.setFont("Helvetica-Bold" if bold else "Helvetica", size)
    c.drawString(COL2_X, y, text)
    y -= 11

# ── Column 3: Model performance ──────────────────────────────────────
c.setFillColorRGB(*PRIMARY)
c.setFont("Helvetica-Bold", 10)
c.drawString(COL3_X, COL_TOP, "Model Performance")

metrics = [
    ("ROC-AUC", "0.871",
     "Ranks shows by risk correctly\n87% of the time."),
    ("Precision", "0.53",
     "When it flags a show at risk,\nit's right 53% of the time."),
    ("Recall", "0.65",
     "Catches 65% of shows that\nwill actually be cancelled."),
]

y = COL_TOP - 28
for label, value, explanation in metrics:
    c.setFillColorRGB(*SECONDARY)
    c.setFont("Helvetica-Bold", 7)
    c.drawString(COL3_X, y, label.upper())
    y -= 16

    c.setFillColorRGB(*ACCENT)
    c.setFont("Helvetica-Bold", 16)
    c.drawString(COL3_X, y, value)
    y -= 18

    c.setFillColorRGB(*TEXT)
    c.setFont("Helvetica", 7)
    for line in explanation.split("\n"):
        c.drawString(COL3_X, y, line)
        y -= 10
    y -= 10


# ── Footer ───────────────────────────────────────────────────────────
c.setStrokeColorRGB(*SECONDARY)
c.line(MARGIN, 42, PAGE_W - MARGIN, 42)
c.setFillColorRGB(*SECONDARY)
c.setFont("Helvetica", 7)
c.drawRightString(PAGE_W - MARGIN, 30, "github.com/jtambe007")


c.showPage()
c.save()

